# Bước 1: Xác minh Dataset Paths trên Kaggle

Script này giúp xác minh:
1. Đường dẫn metadata JSON
2. Đường dẫn thư mục ảnh
3. Format của labels
4. Metadata fields (weather, scene, timeofday)

In [ ]:
import os
import json
from pathlib import Path

## 1. Tìm đường dẫn Dataset

In [ ]:
# Kaggle dataset base path
DATASET_BASE = "/kaggle/input/datasets/solesensei/solesensei_bdd100k"

print("Dataset base:", DATASET_BASE)
print("Exists:", os.path.exists(DATASET_BASE))

# Liệt kê cấu trúc thư mục
def show_tree(path, prefix="", max_depth=3, current_depth=0):
    if current_depth >= max_depth:
        return
    
    path = Path(path)
    if not path.exists():
        print(f"{prefix}❌ {path.name} (NOT FOUND)")
        return
    
    if path.is_file():
        size = os.path.getsize(path) / (1024*1024)  # MB
        print(f"{prefix}📄 {path.name} ({size:.2f} MB)")
    else:
        print(f"{prefix}📁 {path.name}/")
        try:
            items = sorted(path.iterdir())
            for item in items[:10]:  # Giới hạn 10 items
                show_tree(item, prefix + "   ", max_depth, current_depth + 1)
            if len(items) > 10:
                print(f"{prefix}   ... và {len(items) - 10} items khác")
        except PermissionError:
            print(f"{prefix}   [Permission denied]")

show_tree(DATASET_BASE)

## 2. Tìm file JSON labels

In [ ]:
# Tìm tất cả file .json có chứa "labels"
def find_json_files(base_path):
    json_files = []
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if f.endswith(".json"):
                full_path = os.path.join(root, f)
                json_files.append(full_path)
    return json_files

json_files = find_json_files(DATASET_BASE)
print(f"Tìm thấy {len(json_files)} file JSON:")
for f in json_files:
    print(f"  - {f}")

## 3. Tìm thư mục chứa ảnh

In [ ]:
# Tìm thư mục chứa ảnh
def find_image_dirs(base_path):
    image_dirs = []
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_dirs.append(root)
                break
    return list(set(image_dirs))

image_dirs = find_image_dirs(DATASET_BASE)
print(f"Tìm thấy {len(image_dirs)} thư mục chứa ảnh:")
for d in image_dirs[:10]:
    count = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"  - {d} ({count} ảnh)")

## 4. Xác minh Metadata JSON

In [ ]:
# Chọn file JSON đầu tiên tìm được
METADATA_PATH = None
for f in json_files:
    if "train" in f.lower() and "labels" in f.lower():
        METADATA_PATH = f
        break

if not METADATA_PATH:
    METADATA_PATH = json_files[0] if json_files else None

if METADATA_PATH:
    print(f"Sử dụng metadata: {METADATA_PATH}")
    
    # Load và xem sample
    with open(METADATA_PATH, 'r') as f:
        metadata = json.load(f)
    
    print(f"\nTổng số records: {len(metadata)}")
    print(f"\nSample record (record 0):")
    print(json.dumps(metadata[0], indent=2)[:2000])
else:
    print("❌ Không tìm thấy file metadata!")

## 5. Xác minh Metadata Fields

In [ ]:
if METADATA_PATH:
    from collections import Counter
    
    # Lấy 500 sample đầu để phân tích nhanh
    sample_size = 500
    sample_data = metadata[:sample_size]
    
    print(f"Phân tích {sample_size} records đầu tiên:\n")
    
    # Weather
    weather_values = [r.get('attributes', {}).get('weather', 'MISSING') for r in sample_data]
    print("Weather values:")
    for k, v in Counter(weather_values).most_common():
        print(f"  {k}: {v}")
    
    # Scene
    print("\nScene values:")
    scene_values = [r.get('attributes', {}).get('scene', 'MISSING') for r in sample_data]
    for k, v in Counter(scene_values).most_common():
        print(f"  {k}: {v}")
    
    # Timeofday
    print("\nTimeofday values:")
    timeofday_values = [r.get('attributes', {}).get('timeofday', 'MISSING') for r in sample_data]
    for k, v in Counter(timeofday_values).most_common():
        print(f"  {k}: {v}")

## 6. Xác minh Labels Format

In [ ]:
if METADATA_PATH:
    # Kiểm tra xem metadata có chứa labels không
    sample = metadata[0]
    
    print("Kiểm tra labels trong metadata:")
    
    if 'labels' in sample:
        print(f"✅ Metadata có chứa 'labels' field")
        print(f"   Số labels trong record 0: {len(sample['labels'])}")
        
        if sample['labels']:
            print(f"\n   Sample label:")
            print(json.dumps(sample['labels'][0], indent=2))
    else:
        print("❌ Metadata KHÔNG có 'labels' field")
        print("   Cần tìm đường dẫn labels riêng")
    
    # Kiểm tra filename
    print(f"\nFilename sample: {sample.get('name', 'MISSING')}")

## 7. Xác minh Ảnh tồn tại

In [ ]:
if METADATA_PATH and image_dirs:
    # Thử ghép đường dẫn ảnh
    sample = metadata[0]
    filename = sample.get('name', '')
    
    print(f"Sample filename: {filename}")
    
    # Thử các đường dẫn khác nhau
    possible_paths = []
    for img_dir in image_dirs:
        possible_paths.append(os.path.join(img_dir, filename))
    
    print("\nKiểm tra đường dẫn ảnh:")
    for p in possible_paths:
        exists = os.path.exists(p)
        status = "✅ EXISTS" if exists else "❌ NOT FOUND"
        print(f"  {status}: {p}")
    
    # Đếm số ảnh tồn tại trong sample
    existing_count = 0
    for record in metadata[:100]:  # Check 100 records
        fname = record.get('name', '')
        for img_dir in image_dirs:
            if os.path.exists(os.path.join(img_dir, fname)):
                existing_count += 1
                break
    
    print(f"\nTrong 100 records đầu:")
    print(f"  Ảnh tồn tại: {existing_count}/100")

## 8. Tổng kết

In [ ]:
print("=" * 60)
print("KẾT QUẢ XÁC MINH DATASET")
print("=" * 60)

summary = {}

# Metadata
summary['metadata_path'] = METADATA_PATH if METADATA_PATH else "❌ NOT FOUND"
summary['total_records'] = len(metadata) if METADATA_PATH else 0

# Images
summary['image_dirs'] = image_dirs if image_dirs else ["❌ NOT FOUND"]

# Labels
has_labels = 'labels' in metadata[0] if METADATA_PATH else False
summary['has_labels_in_metadata'] = "✅ YES" if has_labels else "❌ NO"

for k, v in summary.items():
    print(f"{k}: {v}")

print("\n" + "=" * 60)
print("CẬP NHẬT ĐƯỜNG DẪN TRONG NOTEBOOK")
print("=" * 60)
if METADATA_PATH:
    print(f'METADATA_PATH = "{METADATA_PATH}"')
if image_dirs:
    print(f'IMAGE_DIR = "{image_dirs[0]}"')